# Agentic AI Systems — Multi‑Agent Fact Checker (CrewAI)

**Theme:** Agentic AI Systems  
**Goal:** A multi‑agent system that performs **fact checking** through **planning, tool usage, and collaboration**.

## What we’ll demo
- **Evidence Hunter** uses a tool (Wikipedia API) to retrieve candidate evidence.
- **Entailment Judge** decides: `SUPPORTED / REFUTED / NOT ENOUGH INFO`.
- **Evidence Auditor** critiques evidence quality and can trigger **autonomous retries**.
- Batch evaluation on a small claim set + a printable trace.

---

## Architecture (high-level)

```
User Claim
   |
   v
[Orchestrator]
   |
   +--> [Evidence Hunter] --(Wikipedia tool)--> Evidence Candidates
   |
   +--> [Entailment Judge] -------------------> Label + Selected Evidence
   |
   +--> [Evidence Auditor] --------------------> PASS / RETRY (+ hint)
                      |
                      +---- if RETRY: loop back to Hunter (with hint)
```

> Note: This notebook uses Wikipedia as a lightweight “evidence store” so the system demonstrates *tool usage* clearly.


In [3]:
# Install dependencies (Colab)
!pip -q install crewai crewai-tools langchain-openai rank-bm25 nltk

import os, re, json, time, requests
from typing import List, Dict, Any, Optional
from rank_bm25 import BM25Okapi

import nltk
nltk.download("punkt", quiet=True)
from nltk.tokenize import sent_tokenize

from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI


## 1) Set API Key

In Google Colab:
- **Runtime → Secrets →** add `OPENAI_API_KEY`

Or set it manually (not recommended):
```python
os.environ["OPENAI_API_KEY"] = "..."
```


In [6]:
# --- API Key check ---
assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY. Add it in Colab Secrets or os.environ."

# LLM (deterministic)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


## 2) Tooling: Wikipedia Evidence Retriever

This tool does:
1) Wikipedia search for candidate pages  
2) Download page extracts  
3) Split into sentences  
4) Rank sentences vs the claim using BM25  
5) Return top sentences as evidence candidates


In [7]:
WIKI_API = "https://en.wikipedia.org/w/api.php"

def _clean_text(t: str) -> str:
    t = re.sub(r"<.*?>", "", t)  # remove html tags from snippets
    t = re.sub(r"\s+", " ", t).strip()
    return t

def _tokenize(text: str) -> List[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

def wiki_search(query: str, limit: int = 6) -> List[Dict[str, Any]]:
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "utf8": 1,
        "srlimit": limit,
    }
    r = requests.get(WIKI_API, params=params, timeout=20)
    r.raise_for_status()
    data = r.json()
    results = []
    for item in data.get("query", {}).get("search", []):
        results.append({
            "title": item["title"],
            "snippet": _clean_text(item.get("snippet",""))
        })
    return results

def wiki_extract(title: str, char_limit: int = 12000) -> str:
    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "titles": title,
        "format": "json",
        "utf8": 1,
        "exsectionformat": "plain",
    }
    r = requests.get(WIKI_API, params=params, timeout=20)
    r.raise_for_status()
    pages = r.json().get("query", {}).get("pages", {})
    page = next(iter(pages.values()), {})
    text = _clean_text(page.get("extract", ""))
    return text[:char_limit]

def rank_sentences_bm25(claim: str, titles: List[str], top_k: int = 10) -> List[Dict[str, str]]:
    pool = []
    for t in titles:
        txt = wiki_extract(t)
        if not txt:
            continue
        for s in sent_tokenize(txt):
            s = s.strip()
            # Filter very short or very long sentences
            if 30 <= len(s) <= 400:
                pool.append({"title": t, "sentence": s})

    if not pool:
        return []

    corpus = [_tokenize(x["sentence"]) for x in pool]
    bm25 = BM25Okapi(corpus)
    scores = bm25.get_scores(_tokenize(claim))
    idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [pool[i] for i in idx]

@tool("wikipedia_evidence_tool")
def wikipedia_evidence_tool(claim: str) -> str:
    """
    Search Wikipedia for a claim and return top evidence sentences (ranked).
    Returns JSON string:
    {
      "claim": "...",
      "pages": [{"title": "...", "snippet": "..."}, ...],
      "evidence": [{"title": "...", "sentence": "..."}, ...]
    }
    """
    pages = wiki_search(claim, limit=6)
    titles = [p["title"] for p in pages]
    evidence = rank_sentences_bm25(claim, titles, top_k=10)
    return json.dumps({"claim": claim, "pages": pages, "evidence": evidence}, ensure_ascii=False)


## 3) Agents (CrewAI roles)

- **Evidence Hunter**: tool usage + retrieval
- **Entailment Judge**: label decision from evidence
- **Evidence Auditor**: checks relevance/sufficiency, requests retry when needed
- **Orchestrator**: lightweight Python loop that repeats if Auditor says RETRY


In [8]:
evidence_hunter = Agent(
    role="Evidence Hunter",
    goal="Use tools to retrieve strong Wikipedia evidence sentences relevant to the claim.",
    backstory="You are a retrieval specialist. You prioritize direct, specific evidence.",
    tools=[wikipedia_evidence_tool],
    llm=llm,
    verbose=True,
)

entailment_judge = Agent(
    role="Entailment Judge",
    goal="Given a claim and evidence, decide SUPPORTED, REFUTED, or NOT ENOUGH INFO.",
    backstory="You are a careful verifier. You do not guess beyond the provided evidence.",
    llm=llm,
    verbose=True,
)

evidence_auditor = Agent(
    role="Evidence Auditor",
    goal="Audit the judge's label and evidence for relevance, sufficiency, and consistency.",
    backstory="You are strict and detail-oriented. You request retries if evidence is weak.",
    llm=llm,
    verbose=True,
)


## 4) Structured Tasks (JSON outputs)

Each agent returns **JSON only** so we can parse and evaluate automatically.


In [9]:
def build_tasks(claim: str, evidence_min: int = 6, evidence_max: int = 10):
    t1 = Task(
        description=f'''
CLAIM: "{claim}"

Use wikipedia_evidence_tool to retrieve evidence candidates.
Return ONLY JSON with this schema:
{{
  "claim": "...",
  "pages": [{{"title":"...", "snippet":"..."}}, ...],
  "evidence": [{{"title":"...", "sentence":"..."}}, ...]
}}

Rules:
- You MUST call the tool at least once.
- Return between {evidence_min} and {evidence_max} evidence items.
- Evidence should be as direct as possible (definitions, factual sentences, etc.).
'''.strip(),
        expected_output="JSON with claim, pages, evidence.",
        agent=evidence_hunter,
    )

    t2 = Task(
        description='''
You will receive JSON from Evidence Hunter.

Decide the label:
- SUPPORTED: evidence clearly supports the claim
- REFUTED: evidence clearly contradicts the claim
- NOT ENOUGH INFO: evidence is missing, too weak, or ambiguous

Return ONLY JSON with this schema:
{
  "label": "SUPPORTED|REFUTED|NOT ENOUGH INFO",
  "selected_evidence": [{"title":"...", "sentence":"..."}, ...],  // 1 to 4 items
  "rationale": "2-5 sentences, based ONLY on selected evidence."
}

Rules:
- If label is SUPPORTED or REFUTED, you must choose evidence that directly supports that decision.
- If label is NOT ENOUGH INFO, explain what is missing.
'''.strip(),
        expected_output="JSON verdict with label + selected evidence + rationale.",
        agent=entailment_judge,
        context=[t1],
    )

    t3 = Task(
        description='''
Audit the judge output for:
1) Relevance: selected evidence must be directly about the claim
2) Sufficiency: enough to justify SUPPORTED/REFUTED
3) Consistency: rationale matches evidence

If evidence is weak or irrelevant, set status=RETRY and provide a better search hint.
Otherwise status=PASS.

Return ONLY JSON:
{
  "status": "PASS|RETRY",
  "issues": ["..."],
  "retry_hint": "..."   // only when RETRY
}
'''.strip(),
        expected_output="JSON audit PASS/RETRY + issues.",
        agent=evidence_auditor,
        context=[t2],
    )

    return t1, t2, t3


## 5) Orchestrator: autonomous retry loop

- Runs Hunter → Judge → Auditor sequentially
- If Auditor says **RETRY**, it reruns with a hint appended to the claim query
- Returns a structured trace with parsed outputs


In [10]:
import os
os.environ["CREWAI_TRACING_ENABLED"] = "true"

def _extract_json_objects(text: str) -> List[dict]:
    # Best-effort JSON extraction: find {...} blocks and parse
    # Works well because we force agents to output JSON only.
    candidates = re.findall(r"\{(?:[^{}]|\{[^{}]*\})*\}", text, flags=re.DOTALL)
    objs = []
    for c in candidates:
        try:
            objs.append(json.loads(c))
        except Exception:
            pass
    return objs

def run_fact_check(claim: str, max_rounds: int = 2, sleep_s: float = 0.0) -> Dict[str, Any]:
    trace = {"claim": claim, "rounds": []}
    hint: Optional[str] = None

    for r in range(1, max_rounds + 1):
        query = claim if not hint else f"{claim}. Retrieval hint: {hint}"

        t1, t2, t3 = build_tasks(query)

        crew = Crew(
            agents=[evidence_hunter, entailment_judge, evidence_auditor],
            tasks=[t1, t2, t3],
            process=Process.sequential,
            verbose=True,
        )

        result = crew.kickoff()
        raw = str(result)

        # Parse objects (we expect 3 JSONs: hunter, judge, auditor)
        objs = _extract_json_objects(raw)

        hunter_obj, judge_obj, audit_obj = None, None, None
        # identify them by keys
        for o in objs:
            if isinstance(o, dict) and "pages" in o and "evidence" in o and "claim" in o:
                hunter_obj = o
            elif isinstance(o, dict) and "label" in o and "selected_evidence" in o:
                judge_obj = o
            elif isinstance(o, dict) and "status" in o and "issues" in o:
                audit_obj = o

        round_rec = {
            "round": r,
            "query": query,
            "hunter": hunter_obj,
            "judge": judge_obj,
            "audit": audit_obj,
            "raw": raw
        }
        trace["rounds"].append(round_rec)

        if audit_obj and audit_obj.get("status") == "RETRY":
            hint = audit_obj.get("retry_hint", "Use more specific keywords.")
            if sleep_s:
                time.sleep(sleep_s)
            continue

        break

    # convenience: final decision
    final_judge = trace["rounds"][-1].get("judge") if trace["rounds"] else None
    trace["final_label"] = final_judge.get("label") if final_judge else None
    trace["final_selected_evidence"] = final_judge.get("selected_evidence") if final_judge else None
    trace["final_rationale"] = final_judge.get("rationale") if final_judge else None
    return trace


## 6) Single-claim demo (interactive)

Edit the claim below and run.


In [ ]:
demo_claim = "The Eiffel Tower is located in Berlin."
demo_trace = run_fact_check(demo_claim, max_rounds=2)

print("\n=== FINAL LABEL ===")
print(demo_trace["final_label"])
print("\n=== RATIONALE ===")
print(demo_trace["final_rationale"])
print("\n=== SELECTED EVIDENCE ===")
for i, ev in enumerate(demo_trace.get("final_selected_evidence") or [], 1):
    print(f"{i}. [{ev.get('title')}] {ev.get('sentence')}")


## 7) Batch evaluation (10 claims)

This is a **lightweight classroom evaluation**:
- We provide a small set of claims with an **expected** label.
- The notebook runs the multi-agent system and reports accuracy.

> These are meant for demo/testing, not an official FEVER-score evaluation.


In [12]:
# 10 demo claims with expected labels (for quick grading)
# (Avoid time-based claims; keep them stable.)
demo_set = [
    {"claim": "Paris is the capital of France.", "expected": "SUPPORTED"},
    {"claim": "The Eiffel Tower is located in Berlin.", "expected": "REFUTED"},
    {"claim": "The Nile is the longest river in the world.", "expected": "SUPPORTED"},  # commonly supported on Wikipedia
    {"claim": "Mount Everest is the tallest mountain above sea level.", "expected": "SUPPORTED"},
    {"claim": "The Great Wall of China is visible from the Moon with the naked eye.", "expected": "REFUTED"},
    {"claim": "Albert Einstein won the Nobel Prize in Physics in 1921.", "expected": "SUPPORTED"},
    {"claim": "The Amazon rainforest is located primarily in Africa.", "expected": "REFUTED"},
    {"claim": "Python is a species of bird.", "expected": "NOT ENOUGH INFO"},  # ambiguous word: programming language vs snake
    {"claim": "India is located in Europe.", "expected": "REFUTED"},
    {"claim": "Coffee was first cultivated in Ethiopia.", "expected": "SUPPORTED"}  # commonly supported on Wikipedia
]


In [ ]:
def evaluate_demo_set(items: List[Dict[str, str]], max_rounds: int = 2) -> Dict[str, Any]:
    results = []
    correct = 0
    for i, it in enumerate(items, 1):
        claim = it["claim"]
        exp = it["expected"]
        print(f"\n\n===== ({i}/{len(items)}) CLAIM =====\n{claim}")
        trace = run_fact_check(claim, max_rounds=max_rounds)
        pred = trace["final_label"] or "UNKNOWN"
        ok = (pred == exp)
        correct += int(ok)
        results.append({
            "claim": claim,
            "expected": exp,
            "predicted": pred,
            "correct": ok,
            "rounds_used": len(trace["rounds"]),
            "evidence_count": len(trace.get("final_selected_evidence") or []),
        })

    acc = correct / len(items) if items else 0.0
    return {"accuracy": acc, "results": results}

demo_eval = evaluate_demo_set(demo_set, max_rounds=2)
print("\n\n=== DEMO ACCURACY ===")
print(demo_eval["accuracy"])


## 8) Pretty print results table


In [ ]:
# Print a compact table
from textwrap import shorten

print(f"{'Correct':7} | {'Exp':16} | {'Pred':16} | Rounds | Claim")
print("-"*90)
for r in demo_eval["results"]:
    print(f"{str(r['correct']):7} | {r['expected'][:16]:16} | {r['predicted'][:16]:16} | {r['rounds_used']:6} | {shorten(r['claim'], width=40)}")


## 9) Export one trace to JSON (for your report appendix)

This lets you attach an example run (full raw output + parsed JSON).


In [ ]:
# Export the last demo trace (or any trace you want)
trace_to_export = demo_trace

with open("example_trace.json", "w", encoding="utf-8") as f:
    json.dump(trace_to_export, f, ensure_ascii=False, indent=2)

print("Saved: example_trace.json")
